In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-5 ; m=1.0 ; a=1.0 ; b=0.5 ; g=9.81 ;
tf=4; N=int(tf/dt);

t=np.linspace(0,tf,N)


theta_0 = 0                                   # condiciones iniciales: θ(0) = 0,  θ̇(0) > 0
thd_0s  = np.array([3.0, 6.0, 9.0])           # varios valores de θ̇(0) para el diagrama de fases
i_main  = 1                                   # caso principal (el que se anima): θ̇(0) = thd_0s[i_main]

## HAMILTONIANO Y ECUACIONES DE HAMILTON (H = p·θ̇ − L)

In [ ]:
#Lagrangiano, momento y Hamiltoniano   (H = p θ̇ − L)
th_s, thd_s, p_s = sp.symbols('theta thetadot p', real=True)
m_, a_, b_, g_ = sp.symbols('m a b g', positive=True)

# Alambre helicoidal:  x = a cosθ ,  y = a sinθ ,  z = b θ     (z hacia arriba)
x_s, y_s, z_s = a_*sp.cos(th_s), a_*sp.sin(th_s), b_*th_s
xd_s, yd_s, zd_s = (sp.diff(f, th_s)*thd_s for f in (x_s, y_s, z_s))

T_sym = sp.simplify(sp.Rational(1,2)*m_*(xd_s**2 + yd_s**2 + zd_s**2))
U_sym = m_*g_*z_s
L_sym = T_sym - U_sym
E_sym = T_sym + U_sym

p_expr = sp.diff(L_sym, thd_s)                      # p = ∂L/∂θ̇   (derivada parcial)
H_vel  = sp.simplify(p_expr*thd_s - L_sym)          # H = p θ̇ − L   (en función de θ, θ̇)

print('T =', T_sym, ' ,  U =', U_sym)
print('p =', p_expr)
print('H(θ, θ̇) = p θ̇ − L =', H_vel)
print('H − E =', sp.simplify(H_vel - E_sym), '   ->  H = E' if sp.simplify(H_vel - E_sym) == 0 else '   ->  H ≠ E')

# Hamiltoniano en variables canónicas: θ̇ = θ̇(p)
sol   = sp.solve(sp.Eq(p_s, p_expr), thd_s, dict=True)[0]
H_can = sp.simplify(H_vel.subs(sol))
print('H(θ, p) =', H_can)

# Ecuaciones de Hamilton
thdot_eq = sp.diff(H_can, p_s)                      #  θ̇ =  ∂H/∂p
pdot_eq  = -sp.diff(H_can, th_s)                    #  ṗ = −∂H/∂θ
print('θ̇ =', thdot_eq, ' ,  ṗ =', pdot_eq)

# θ̈ = d/dt(∂H/∂p)  ->  constante
c_sym = sp.simplify(sp.diff(thdot_eq, th_s)*thdot_eq + sp.diff(thdot_eq, p_s)*pdot_eq)
print('θ̈ = c =', c_sym)

f_num     = sp.lambdify((th_s, p_s, m_, a_, b_, g_), [thdot_eq, pdot_eq], 'numpy')
H_num     = sp.lambdify((th_s, p_s, m_, a_, b_, g_), H_can, 'numpy')
th_orb    = sp.lambdify((p_s, sp.Symbol('E', real=True), m_, a_, b_, g_), sp.solve(sp.Eq(H_can, sp.Symbol('E', real=True)), th_s)[0], 'numpy')

## FUNCION PARA EL MOVIMIENTO

In [ ]:
def Motion(th0, p0, pars, t):
    """Integra las ecuaciones de Hamilton para varios casos a la vez (Euler predictor + Heun corrector)."""
    N, nc = len(t), len(p0)
    th, p = np.zeros((nc, N)), np.zeros((nc, N))
    th[:, 0], p[:, 0] = th0, p0

    for i in range(N - 1):
        dt = t[i+1] - t[i]

        # --- PREDICTOR (Euler) ---
        thd_i, pd_i = f_num(th[:, i], p[:, i], *pars)
        th_pred = th[:, i] + thd_i*dt
        p_pred  = p[:, i]  + pd_i*dt

        # --- Derivadas en el punto predicho ---
        thd_n, pd_n = f_num(th_pred, p_pred, *pars)

        # --- CORRECTOR (Heun) ---
        th[:, i+1] = th[:, i] + 0.5*(thd_i + thd_n)*dt
        p[:, i+1]  = p[:, i]  + 0.5*(pd_i + pd_n)*dt

    return th, p


#Simulación
pars = (m, a, b, g)
M_eff = m*(a**2 + b**2)                               # "masa efectiva" del movimiento en θ
p0s   = M_eff*thd_0s                                  # p(0) = m (a²+b²) θ̇(0)
theta, p = Motion(np.full(len(p0s), float(theta_0)), p0s, pars, t)
thd = p/M_eff

## SOLUCIÓN ANALÍTICA Y ENERGÍA

In [ ]:
#Solución analítica:  θ̈ = c  (constante)  ->  θ̇(t) = θ̇0 + c t ,  θ(t) = θ0 + θ̇0 t + ½ c t²
c = -m*g*b/M_eff                                      # c = −g b/(a² + b²)
th_an  = theta_0 + thd_0s[:, None]*t[None, :] + 0.5*c*t[None, :]**2
thd_an = thd_0s[:, None] + c*t[None, :]
p_an   = M_eff*thd_an

#Energía (H = E se conserva):  E = p²/(2 m (a²+b²)) + m g b θ
E = H_num(theta, p, *pars)

print(f'c = θ̈ = {c:.4f} rad/s²   (c = −g b/(a²+b²))')
print(f'{"θ̇0":>6}{"t_subida (s)":>15}{"θ_max (rad)":>14}{"z_max (m)":>12}{"t_vuelta (s)":>15}{"max|E−E0|":>13}{"max|θ−θ_an|":>14}{"max|p−p_an|":>14}')
for i, w0 in enumerate(thd_0s):
    t_up = w0/abs(c)
    print(f'{w0:>6.2f}{t_up:>15.4f}{w0**2/(2*abs(c)):>14.4f}{b*w0**2/(2*abs(c)):>12.4f}{2*t_up:>15.4f}'
          f'{np.abs(E[i]-E[i,0]).max():>13.2e}{np.abs(theta[i]-th_an[i]).max():>14.2e}{np.abs(p[i]-p_an[i]).max():>14.2e}')

## DIAGRAMA DE FASES

In [ ]:
#Diagrama de fases (θ, p)
t_mark = 0.5*np.arange(0, int(tf/0.5) + 1)             # marcas de tiempo (s)
i_mark = np.clip(np.round(t_mark/dt).astype(int), 0, N - 1)

th_lo, th_hi = theta.min() - 1.0, theta.max() + 1.0
p_lo,  p_hi  = p.min() - 2.0,     p.max() + 2.0
thg, pg = np.linspace(th_lo, th_hi, 300), np.linspace(p_lo, p_hi, 300)
THg, Pg = np.meshgrid(thg, pg)
p_full  = np.linspace(p_lo, p_hi, 600)
cols = ['tab:blue', 'tab:red', 'tab:green']

fig, ax = plt.subplots(figsize=(10, 7))

# Campo de flujo (θ̇ = p/M, ṗ = −m g b)
ax.streamplot(thg, pg, Pg/M_eff, np.full_like(THg, -m*g*b), color='gainsboro', density=1.2, linewidth=0.9, arrowsize=1.2)

for i, w0 in enumerate(thd_0s):
    col, lw = cols[i], (3 if i == i_main else 1.8)
    ax.plot(th_orb(p_full, E[i, 0], *pars), p_full, '--', color=col, lw=1, alpha=0.4)                  # órbita completa
    ax.plot(theta[i], p[i], color=col, lw=lw, label=f'$\\dot\\theta_0 = {w0:g}$ rad/s' + ('  (caso principal)' if i == i_main else ''))
    ax.scatter(theta[i, i_mark], p[i, i_mark], s=18, color='white', edgecolors=col, zorder=4)           # marcas de tiempo
    ax.scatter(theta[i, 0], p[i, 0], s=80, color=col, edgecolors='black', zorder=5)                     # condición inicial
    ax.scatter(w0**2/(2*abs(c)), 0, marker='*', s=140, color=col, edgecolors='black', zorder=5)         # punto de retroceso (p = 0)
    ax.annotate('', xy=(theta[i, -1], p[i, -1]), xytext=(theta[i, -400], p[i, -400]),
                arrowprops=dict(arrowstyle='-|>', color=col, lw=2, mutation_scale=18), zorder=5)         # sentido de recorrido

ax.scatter([], [], s=18, color='white', edgecolors='k', label=f'marcas cada {t_mark[1]:.1f} s')
ax.scatter([], [], marker='*', s=100, color='gray', edgecolors='k', label='punto de retroceso (θ̇ = 0)')
ax.axhline(0, color='k', ls=':', lw=1); ax.axvline(0, color='k', ls=':', lw=1)
ax.set(xlim=(th_lo, th_hi), ylim=(p_lo, p_hi), xlabel='$\\theta$ (rad)', ylabel='$p_\\theta$ (kg m²/s)',
       title='Diagrama de fases: cuenta en un alambre helicoidal')
ax.grid(alpha=0.4, ls='--'); ax.legend(loc='upper left', framealpha=0.9, fontsize=9)
plt.tight_layout(); plt.show()


#θ(t), θ̇(t): numérica frente a analítica
s = max(1, N // 5000)
fig, ax2 = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for i, w0 in enumerate(thd_0s):
    ax2[0,0].plot(t[::s], theta[i, ::s], color=cols[i], lw=2, label=f'$\\dot\\theta_0={w0:g}$')
    ax2[0,0].plot(t[::s], th_an[i, ::s], 'k:', lw=1)
    ax2[0,1].plot(t[::s], thd[i, ::s], color=cols[i], lw=2)
    ax2[0,1].plot(t[::s], thd_an[i, ::s], 'k:', lw=1)
    ax2[1,0].plot(t[::s], (theta[i] - th_an[i])[::s], color=cols[i])
    ax2[1,1].plot(t[::s], (thd[i] - thd_an[i])[::s], color=cols[i])
ax2[0,0].set(ylabel='θ (rad)', title='θ(t)  (punteada: analítica)'); ax2[0,0].legend(fontsize=8)
ax2[0,1].set(ylabel='θ̇ (rad/s)', title='θ̇(t)')
ax2[1,0].set(xlabel='t (s)', ylabel='θ_num − θ_an'); ax2[1,1].set(xlabel='t (s)', ylabel='θ̇_num − θ̇_an')
for A in ax2.ravel(): A.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## ANIMACIÓN (3D + dos planos 2D + espacio de fases)

In [ ]:
#Animación: vista 3D + planos 2D (lateral x-z y cenital x-y) + espacio de fases
from IPython.display import display
plt.rcParams['animation.embed_limit'] = 100        # MB; el límite por defecto (20) se queda corto

interactivo = True     # True : figura interactiva, se gira la vista 3D con el ratón mientras se anima (requiere ipympl)
                       # False: animación HTML (el ratón no gira la vista; usa 'rotar' para que la cámara gire sola)
elev, azim  = 20, -60  # ángulos iniciales de la vista 3D (grados)
rotar       = 0        # giro automático de la cámara (grados por segundo; 0 = fija). Funciona en los dos modos

if interactivo:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')      # backend interactivo:  pip install ipympl
    except Exception as e:
        interactivo = False
        print('Recordatorio para el modo interactivo (instalar ipympl).', e)

T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 15       # fotogramas por segundo
speed   = 0.5      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 2.0      # duración de la estela de la cuenta (s); 0 = sin estela

# Submuestreo: un fotograma cada 'step' pasos de simulación
step    = max(1, round(speed / (fps * dt)))
idx     = np.arange(0, min(N, int(T_anim / dt)), step)
trail_n = int(trail_s / (dt * step))

ths, ps_, ts = theta[i_main, idx], p[i_main, idx], t[idx]
xs, ys, zs = a*np.cos(ths), a*np.sin(ths), b*ths   # posición de la cuenta

# Alambre
th_w = np.linspace(theta[i_main].min() - 0.5, theta[i_main].max() + 0.5, 800)
xw, yw, zw = a*np.cos(th_w), a*np.sin(th_w), b*th_w
Rl = 1.25*a
z0, z1 = zw.min() - 0.1, zw.max() + 0.1

fig = plt.figure(figsize=(12, 9))
ax3 = fig.add_subplot(2, 2, 1, projection='3d')
axs = fig.add_subplot(2, 2, 2)                     # plano lateral x-z
axt = fig.add_subplot(2, 2, 3)                     # plano cenital x-y
axp = fig.add_subplot(2, 2, 4)                     # espacio de fases
fig.subplots_adjust(left=0.06, right=0.97, bottom=0.07, top=0.93, wspace=0.25, hspace=0.3)

# --- Vista 3D ---
ax3.set(xlim=(-Rl, Rl), ylim=(-Rl, Rl), zlim=(z0, z1), xlabel='x', ylabel='y', zlabel='z', title='Vista 3D')
ax3.set_box_aspect((2*Rl, 2*Rl, z1 - z0))
ax3.grid(False)                                    # sin rejilla
ax3.view_init(elev=elev, azim=azim)
ax3.plot(xw, yw, zw, '-', color='gray', lw=2)      # alambre helicoidal
ax3.plot([0, 0], [0, 0], [z0, z1], 'k:', lw=1)     # eje z

# --- Plano lateral (x-z) ---
axs.set(xlim=(-Rl, Rl), ylim=(z0, z1), aspect='equal', xlabel='x (m)', ylabel='z (m)', title='Plano lateral (x-z)')
axs.plot(xw, zw, '-', color='gray', lw=2); axs.grid(alpha=0.3)

# --- Plano cenital (x-y) ---
axt.set(xlim=(-Rl, Rl), ylim=(-Rl, Rl), aspect='equal', xlabel='x (m)', ylabel='y (m)', title='Plano cenital (x-y)')
axt.plot(xw, yw, '-', color='gray', lw=2); axt.grid(alpha=0.3)

# --- Espacio de fases ---
axp.streamplot(thg, pg, Pg/M_eff, np.full_like(THg, -m*g*b), color='gainsboro', density=1.0, linewidth=0.8, arrowsize=1)
axp.plot(th_orb(p_full, E[i_main, 0], *pars), p_full, '--', color='tab:red', lw=1, alpha=0.4)
axp.axhline(0, color='k', ls=':', lw=1); axp.axvline(0, color='k', ls=':', lw=1)
axp.set(xlim=(th_lo, th_hi), ylim=(p_lo, p_hi), xlabel='θ (rad)', ylabel='p_θ (kg m²/s)', title='Espacio de fases')
axp.grid(alpha=0.4, ls='--')

# Artistas
bead3,  = ax3.plot([], [], [], 'o', color='tab:red', ms=9)
trail3, = ax3.plot([], [], [], '-', color='tab:red', lw=1.5, alpha=0.7)
beadS,  = axs.plot([], [], 'o', color='tab:red', ms=9)
trailS, = axs.plot([], [], '-', color='tab:red', lw=1.5, alpha=0.7)
beadT,  = axt.plot([], [], 'o', color='tab:red', ms=9)
trailT, = axt.plot([], [], '-', color='tab:red', lw=1.5, alpha=0.7)
beadP,  = axp.plot([], [], 'o', color='tab:red', mec='black', ms=9)
trailP, = axp.plot([], [], '-', color='tab:red', lw=2.5)

time_txt = fig.text(0.02, 0.96, '')

def update(k):
    j = max(0, k - trail_n)
    bead3.set_data_3d([xs[k]], [ys[k]], [zs[k]])
    trail3.set_data_3d(xs[j:k+1], ys[j:k+1], zs[j:k+1])
    beadS.set_data([xs[k]], [zs[k]]);  trailS.set_data(xs[j:k+1], zs[j:k+1])
    beadT.set_data([xs[k]], [ys[k]]);  trailT.set_data(xs[j:k+1], ys[j:k+1])
    beadP.set_data([ths[k]], [ps_[k]]); trailP.set_data(ths[:k+1], ps_[:k+1])

    if rotar != 0:                                 # solo si se pide giro automático (si no, se respeta la vista del ratón)
        ax3.view_init(elev=elev, azim=azim + rotar*ts[k])

    time_txt.set_text(f't = {ts[k]:.2f} s    θ = {ths[k]:.2f} rad    z = {zs[k]:.2f} m')
    return bead3, trail3, beadS, trailS, beadT, trailT, beadP, trailP, time_txt

anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=False)   # blit=False: necesario con ejes 3D

if interactivo:
    plt.show()                                     # no cerrar la figura: es la que se maneja con el ratón
else:
    plt.close(fig)
    display(HTML(anim.to_jshtml()))